# TF-IDF preparation for SAS Data Maker
Your task: prepare one training-only ZIP. Your colleague generates synthetic tabular rows in Data Maker. This notebook does not upload, generate data, train classifiers or call LLMs.

The revised v2.1 ESI annotations are provisional development labels. Ten cases and synthetic variants do not establish clinical performance. Keep the test set out of Data Maker.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'evals/tfidf_prep.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the EMS_Comms repository.')
print('Repository:', ROOT)
# If dependencies are missing, run this once in this kernel, then restart if requested:
# subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'evals/requirements-ml.txt')], check=True)
import sklearn, openpyxl
print('scikit-learn:', sklearn.__version__, 'openpyxl:', openpyxl.__version__)

## 1. Choose the source and a new output folder
Upload the unchanged workbook into `evals/local/`. Set both acknowledgements only after confirming this is synthetic data and accepting the provisional-label limitation. The default seed fixes the development split. Do not rerun with different seeds to chase better performance.

In [ ]:
WORKBOOK = ROOT / 'evals/local/Annotation_Notebook_V2.1_Clean.xlsx'
OUT = ROOT / 'evals/results/tfidf-v21-01'  # Must be a NEW directory.
MAX_FEATURES = 30
SEED = 20261021
CONFIRM_SYNTHETIC = False
ACKNOWLEDGE_PROVISIONAL_LABELS = False
GROUP_MAP = None  # Optional complete JSON case-ID -> scenario-family mapping.
if not WORKBOOK.is_file():
    raise FileNotFoundError('Upload the unchanged v2.1 workbook and update WORKBOOK.')
print('Source file found; no upload or preparation performed yet.')

In [ ]:
if not (CONFIRM_SYNTHETIC and ACKNOWLEDGE_PROVISIONAL_LABELS):
    raise ValueError('Confirm both acknowledgements in the preceding cell before preparing data.')
command = [sys.executable, '-m', 'evals.tfidf_prep', 'prepare',
           '--workbook', str(WORKBOOK), '--out', str(OUT),
           '--max-features', str(MAX_FEATURES), '--seed', str(SEED),
           '--test-fraction', '0.3', '--confirm-synthetic', '--acknowledge-provisional-labels']
if GROUP_MAP is not None:
    command += ['--group-map', str(GROUP_MAP)]
result = subprocess.run(command, cwd=ROOT, check=True, capture_output=True, text=True)
print(result.stdout)

In [ ]:
audit = json.loads((OUT / 'preparation_audit.json').read_text())
print({key: audit[key] for key in ['training_rows', 'test_rows', 'features', 'absent_esi_levels']})
print('Send only:', OUT / 'colleague_handoff.zip')
print('Keep private/ and preparation_audit.json with you. Do not upload test rows.')
for warning in audit['warnings']:
    print('-', warning)

## 2. Send only the ZIP
Download `colleague_handoff.zip` from the output folder using Jupyter's file browser. It contains the training CSV, dictionary, schema, manifest and colleague instructions.

Ask for a jointly generated feature/ESI table with identical columns, generation metadata containing the source bundle ID, and the Data Maker quality report. No independent label generation, new ESI classes or test-data use. Do not interpret feature weights as vitals or reconstruct patient text from them.

Keep `private/` and the audit local. When synthetic rows return, the next task is provenance/schema validation and original-only versus augmented ML training on the same fixed original test set. This notebook deliberately does not inspect model performance.

See `evals/TFIDF_DATA_MAKER.md` for commands, limitations and the separate v2.1 LLM-contract requirement.